# railroad · the language

Three things carry everything else: a **fluent** (a fact), a **state**
(facts, a clock, and a queue of facts that have not happened yet), and a
**transition** (dispatch an action, run the world forward). Concurrency falls
out of those semantics rather than out of the planner, which is the thing
worth seeing before anything else.

Run the cells in order, and edit them — that is what this file is for.
Nothing here plans; the planner, the environments and the operator library
all sit on top of what follows.

The rest of the tutorial is an ordinary script you edit and run at the
command line:

```
uv run railroad tutorial          # what step you are on, and what to type
```

In [ ]:
from railroad.core import (
    Action,
    Effect,
    Fluent as F,
    GroundedEffect,
    Operator,
    State,
    get_action_by_name,
    get_next_actions,
    transition,
)

## A fluent is a fact

A predicate and its arguments, and nothing else — no truth value, no time.
The string form is a convenience for typing; both spellings build the same
object, so either is fine wherever you find one easier to read.

In [ ]:
F("at r1 roomA") == F("at", "r1", "roomA")

Negation is `~`, and it is part of the fluent rather than something wrapped
around it. A negative fluent in an effect means *delete this fact*.

In [ ]:
~F("at r1 roomA"), ~F("at r1 roomA") == F("not at r1 roomA")

## A state is fluents + a clock + a queue

The first two are what a PDDL state usually is. The queue —
`upcoming_effects` — is the addition that the whole system turns on: it lets
a state describe a world in which something is *already happening*.

In [ ]:
state = State(0.0, {F("at r1 roomA"), F("free r1"),
                    F("at r2 roomA"), F("free r2")})
state

## An action is a list of effects *at times*

Not a single instant. A move takes both halves: at `t=0` the robot stops
being free and stops being anywhere, and at `t=5` both facts come back, at
the destination.

Notice there is no `moving` predicate. Being nowhere and not free *is* what
moving is, and it is the queue that remembers the arrival.

In [ ]:
move_r1 = Action(
    preconditions={F("at r1 roomA"), F("free r1")},
    effects=[
        GroundedEffect(0.0, {~F("free r1"), ~F("at r1 roomA")}),
        GroundedEffect(5.0, {F("free r1"), F("at r1 roomB")}),
    ],
    name="move r1 roomA roomB",
)

for effect in move_r1.effects:
    print(effect)

In [ ]:
state.satisfies_precondition(move_r1)

## An operator is that action, lifted

Parameters carry a type, and `instantiate` grounds out one action per legal
binding — which is exactly the shape you just wrote by hand. This is the
PDDL half of the interface, and it is all the planner ever sees.

In [ ]:
move = Operator(
    name="move",
    parameters=[("?r", "robot"), ("?from", "location"), ("?to", "location")],
    preconditions=[F("at ?r ?from"), F("free ?r")],
    effects=[
        Effect(time=0, resulting_fluents={~F("free ?r"), ~F("at ?r ?from")}),
        Effect(time=5.0, resulting_fluents={F("free ?r"), F("at ?r ?to")}),
    ],
)

actions = move.instantiate({"robot": ["r1", "r2"],
                           "location": ["roomA", "roomB"]})
[action.name for action in actions]

## Dispatching does not advance the clock

This is the cell to slow down on. `transition` applies the action's `t=0`
effects, queues the rest, and then runs the world forward **only until
somebody is free to act again**.

In [ ]:
after_r1, probability = transition(
    state, get_action_by_name(actions, "move r1 roomA roomB")
)[0]
after_r1

The clock is still `0`: `r2` is free, so nothing *has* to happen yet. `r1` is
neither free nor anywhere, and its arrival sits in the queue with a
timestamp on it.

That is where concurrency comes from. The next decision is made in a world
where one robot is mid-move — no plan-merging step, no separate scheduler.

In [ ]:
[action.name for action in get_next_actions(after_r1, actions)]

Send `r2` after it and nobody is free any more, so now the world has to move
— straight to `t=5`, where the queue comes due.

In [ ]:
after_r2, probability = transition(
    after_r1, get_action_by_name(actions, "move r2 roomA roomB")
)[0]
after_r2

## Effects may branch

Searching a room takes three seconds and *might* turn up the cup. The branch
hangs off the effect that fires at `t=3`, so the uncertainty resolves when
the search finishes rather than when it starts.

One robot this time: with `r2` around, `transition` would stop at `t=0` with
the search still queued, and there would be nothing to see yet.

In [ ]:
search = Action(
    preconditions={F("at r1 roomA"), F("free r1")},
    effects=[
        GroundedEffect(0.0, {~F("free r1")}),
        GroundedEffect(
            3.0,
            {F("free r1"), F("searched roomA cup")},
            prob_effects=[
                (0.8, [GroundedEffect(0.0, {F("found cup"), F("at cup roomA")})]),
                (0.2, []),
            ],
        ),
    ],
    name="search r1 roomA cup",
)

solo = State(0.0, {F("at r1 roomA"), F("free r1")})
for outcome, probability in transition(solo, search):
    print(f"p={probability:.1f}  {outcome}")

`transition` returned a *distribution* rather than a state. Everything
downstream — the planner especially — is written against that list, and the
deterministic case above was simply a list of one.

## A goal is a sentence about fluents

`&`, `|` and `~`, evaluated against a state's facts. Negation is
negation-as-absence: the fluent is simply not there.

In [ ]:
goal = F("at r1 roomB") & ~F("at r2 roomA")
goal, goal.evaluate(after_r2.fluents)

## Where this goes next

That is the language. Everything above is what the planner searches over,
and everything the rest of the tutorial adds — a real problem, a second
robot, hidden objects, a house — is written in exactly these pieces.

```
uv run railroad tutorial          # where you are, and what to type next
uv run railroad tutorial run      # step 01: the same ideas, with a planner
```

Two things you would otherwise write by hand are worth knowing about before
you go: `railroad.operators` builds the operators above (`move`, `pick`,
`place`, `search`) from a few numbers, and `SymbolicEnvironment` runs the
dispatch loop for you. The tutorial writes them out longhand anyway, because
the point is to see them.